## Audit and build temporal parsing and phrasing

In [2]:
import pandas as pd
import re
import random

df = pd.read_parquet("../data/arf_chunks_parsed.parquet")

# chunk_id loaded as string in a past session (Week 4 finding) —
# verify rather than assume it's still an issue, cast if needed.
print(f"chunk_id dtype: {df['chunk_id'].dtype}")
if df["chunk_id"].dtype != "int64":
    df["chunk_id"] = df["chunk_id"].astype(int)

# Random sample across the WHOLE corpus, not just book 106 — the
# point is checking generality, and 106 already biases toward "has
# chapter markers" since we know that from earlier sessions.
random.seed(42)
all_book_ids = df["book_id"].unique().tolist()
SAMPLE_SIZE = 20
sample_books = random.sample(all_book_ids, min(SAMPLE_SIZE, len(all_book_ids)))
print(f"Sampled {len(sample_books)} books: {sample_books}")

# Chapter-marker regex: roman numerals, arabic digits, or common
# spelled-out cardinals/ordinals. Word-bounded and case-insensitive.
# First-pass heuristic, not guaranteed exhaustive — some books may use
# formats this misses (e.g. "I." alone, no "CHAPTER" word); worth
# noting as a limitation rather than assuming full coverage.
NUMBER_WORDS = (
    "ONE|TWO|THREE|FOUR|FIVE|SIX|SEVEN|EIGHT|NINE|TEN|ELEVEN|TWELVE|"
    "THIRTEEN|FOURTEEN|FIFTEEN|SIXTEEN|SEVENTEEN|EIGHTEEN|NINETEEN|TWENTY|"
    "FIRST|SECOND|THIRD|FOURTH|FIFTH|SIXTH|SEVENTH|EIGHTH|NINTH|TENTH"
)
CHAPTER_RE = re.compile(
    rf"\bCHAPTER\s+([IVXLCDM]+|\d+|{NUMBER_WORDS})\b", re.IGNORECASE
)

results = []
for book_id in sample_books:
    book_df = df[df["book_id"] == book_id].sort_values("chunk_id").reset_index(drop=True)

    # Matches WITHIN each chunk's own text. Multiple matches in one
    # chunk = TOC-listing signature (confirmed pattern from book 106's
    # front matter), not a real chapter start.
    per_chunk_matches = book_df["chunk"].apply(
        lambda t: CHAPTER_RE.findall(t) if isinstance(t, str) else []
    )
    match_counts = per_chunk_matches.apply(len)

    toc_like_chunks = book_df[match_counts > 1]
    single_match_chunks = book_df[match_counts == 1].copy()
    single_match_chunks["marker"] = per_chunk_matches[match_counts == 1].apply(lambda m: m[0])

    # Spacing between single-match candidates — TOC listings cluster
    # with tiny gaps; real chapter starts should space out roughly
    # proportionally to chapter length.
    gaps = single_match_chunks["chunk_id"].diff().dropna().tolist()

    results.append({
        "book_id": book_id,
        "n_chunks": len(book_df),
        "toc_like_chunks": len(toc_like_chunks),
        "single_match_candidates": len(single_match_chunks),
        "median_gap": pd.Series(gaps).median() if gaps else None,
        "min_gap": min(gaps) if gaps else None,
        "max_gap": max(gaps) if gaps else None,
    })

summary = pd.DataFrame(results)
print(summary.to_string())

print(f"\nBooks with >=1 candidate marker: {(summary['single_match_candidates'] > 0).sum()} / {len(summary)}")
print(f"Books with >=3 candidates (plausible real chapter structure): {(summary['single_match_candidates'] >= 3).sum()} / {len(summary)}")

chunk_id dtype: str
Sampled 20 books: ['73548', '21299', '12807', '9909', '36684', '34025', '32543', '23060', '18873', '77', '619', '16630', '6941', '5111', '1329', '31858', '3322', '5658', '70653', '64264']
   book_id  n_chunks  toc_like_chunks  single_match_candidates  median_gap  min_gap  max_gap
0    73548       344                2                       19        18.0      1.0     34.0
1    21299      1700                0                       39        31.0      1.0    107.0
2    12807       765                0                       31        27.5      1.0     56.0
3     9909       293                0                       17        16.5      1.0     41.0
4    36684       833                0                       30        32.0      1.0     48.0
5    34025       733                0                        3       319.5      1.0    638.0
6    32543       692                0                       21        31.0      1.0     62.0
7    23060       142                0           

In [3]:
for book_id in summary.sort_values("single_match_candidates", ascending=False)["book_id"].head(3):
    book_df = df[df["book_id"] == book_id].sort_values("chunk_id").reset_index(drop=True)
    per_chunk_matches = book_df["chunk"].apply(
        lambda t: CHAPTER_RE.findall(t) if isinstance(t, str) else []
    )
    match_counts = per_chunk_matches.apply(len)
    single_match_chunks = book_df[match_counts == 1].copy()
    single_match_chunks["marker"] = per_chunk_matches[match_counts == 1].apply(lambda m: m[0])
    print(f"\n=== Book {book_id} ===")
    print(single_match_chunks[["chunk_id", "marker"]].to_string(index=False))


=== Book 5658 ===
 chunk_id marker
       10      1
       11      1
       40      2
       56      3
       57      3
       86      4
       87      4
      106      5
      195      6
      294      7
      344      8
      345      8
      401      9
      461     10
      462     10
      567     11
      568     11
      598     12
      649     13
      714     14
      793     15
      811     16
      812     16
      843     17
      868     18
      928     19
      929     19
      956     20
     1019     21
     1020     21
     1049     22
     1075     23
     1076     23
     1124     24
     1157     25
     1196     26
     1224     27
     1260     28
     1297     29
     1298     29
     1318     30
     1346     31
     1347     31
     1387     32
     1388     32
     1419     33
     1487     34
     1539     35
     1569     36
     1594     37
     1640     38
     1641     38
     1670     39
     1701     40
     1702     40
     1746     41
     1775   

In [4]:
# For the 8 zero-marker books: is it "no chapters exist" or
# "this regex misses the format"? Spot-check raw text near the start
# of each for anything chapter-like the regex didn't catch.
zero_marker_books = summary[summary["single_match_candidates"] == 0]["book_id"].tolist()

for book_id in zero_marker_books:
    book_df = df[df["book_id"] == book_id].sort_values("chunk_id").reset_index(drop=True)
    first_chunks_text = " ".join(book_df["chunk"].head(5).tolist())
    print(f"=== Book {book_id} (first ~5 chunks) ===")
    print(first_chunks_text[:300], "...\n")

=== Book 23060 (first ~5 chunks) ===
***  THE UNKNOWN MASTERPIECE ***




Produced by David Widger





THE UNKNOWN MASTERPIECE

By Honor\xc3\xa9 De Balzac

TO A LORD

1845




I--GILLETTE

On a cold December morning in the year 1612, a young man, whose clothing was somewhat of the thinnest, was walking to and f ...

=== Book 18873 (first ~5 chunks) ===
***  CONTES ET L\xc3\x89GENDES. 1RE PARTIE ***




Produced by Chuck Greif, Jason Isbell and the Online Distributed Proofreading Team at http://www.pgdp.net









CONTES ET L\xc3\x89GENDES

1\xc3\x88RE PARTIE

PAR

H. A. GUERBER

AUTEUR DE "MYTHS OF GREECE AND ROME"

NEW  ...

=== Book 77 (first ~5 chunks) ===
***  THE HOUSE OF THE SEVEN GABLES ***




The House of the Seven Gables

by Nathaniel Hawthorne

With an introduction by George Parsons Lathrop


Contents

 INTRODUCTORY NOTE   AUTHOR'S PREFACE

 I. THE OLD PYNCHEON FAMILY  II. THE LITTLE SHOP-WINDOW  III. THE FIRST CUSTOMER  IV. TH ...

=== Book 16630 (first ~5 chunks) ===
**

In [5]:
FRENCH_MARKERS = {"le", "la", "les", "des", "une", "un", "et", "de", "du",
                   "dans", "que", "pour", "avec", "est", "elle", "il"}
GERMAN_MARKERS = {"der", "die", "das", "und", "ist", "nicht", "mit", "ein", "eine"}
SPANISH_MARKERS = {"el", "la", "los", "las", "que", "para", "con", "una"}

def guess_language(text: str) -> str:
    words = set(re.findall(r"[a-zà-ÿ]+", text.lower()))
    fr, de, es = len(words & FRENCH_MARKERS), len(words & GERMAN_MARKERS), len(words & SPANISH_MARKERS)
    if fr >= 4 and fr >= de and fr >= es:
        return "likely_french"
    if de >= 4:
        return "likely_german"
    if es >= 4:
        return "likely_spanish"
    return "likely_english"

# Corpus-wide, not just today's 20-book sample — this question is
# bigger than chapter parsing, deserves a full check.
lang_rows = []
for book_id, book_df in df.groupby("book_id"):
    first_text = " ".join(book_df.sort_values("chunk_id")["chunk"].head(3).tolist())
    lang_rows.append({"book_id": book_id, "title": book_df["title"].iloc[0],
                       "guess": guess_language(first_text)})

lang_df = pd.DataFrame(lang_rows)
print(lang_df["guess"].value_counts())
print("\nNon-English candidates:")
print(lang_df[lang_df["guess"] != "likely_english"][["book_id", "title", "guess"]])

guess
likely_english    96
Name: count, dtype: int64

Non-English candidates:
Empty DataFrame
Columns: [book_id, title, guess]
Index: []


above number isn't safe to trust

In [6]:
def guess_language(text: str) -> str:
    words = set(re.findall(r"[a-zà-ÿ]+", text.lower()))
    fr, de, es = len(words & FRENCH_MARKERS), len(words & GERMAN_MARKERS), len(words & SPANISH_MARKERS)
    if fr >= 4 and fr >= de and fr >= es:
        return "likely_french"
    if de >= 4:
        return "likely_german"
    if es >= 4:
        return "likely_spanish"
    return "likely_english"

lang_rows = []
for book_id, book_df in df.groupby("book_id"):
    book_df = book_df.sort_values("chunk_id")
    mid = len(book_df) // 2
    # 10 chunks centered on the book's midpoint — well clear of front
    # matter/TOC on either end, per everything learned so far.
    window = book_df.iloc[max(0, mid - 5):mid + 5]
    mid_text = " ".join(window["chunk"].tolist())
    lang_rows.append({"book_id": book_id, "title": book_df["title"].iloc[0],
                       "guess": guess_language(mid_text)})

lang_df = pd.DataFrame(lang_rows)
print(lang_df["guess"].value_counts())
print("\nNon-English candidates:")
print(lang_df[lang_df["guess"] != "likely_english"][["book_id", "title", "guess"]])

# Sanity check against the one case we already know the true answer
# for — don't trust the corpus-wide number until this specific row
# comes back correct.
print("\nBook 18873 (known French, ground truth check):")
print(lang_df[lang_df["book_id"] == "18873"])

guess
likely_english    95
likely_french      1
Name: count, dtype: int64

Non-English candidates:
   book_id                           title          guess
13   18873  Contes et légendes. 1re Partie  likely_french

Book 18873 (known French, ground truth check):
   book_id                           title          guess
13   18873  Contes et légendes. 1re Partie  likely_french


In [10]:
import pandas as pd
import ast
import sys
sys.path.append("..")  # adjust if temporal/ isn't one level up from notebooks/
from temporal.phrase_parsing import parse_temporal_phrase, resolve_to_bins

df = pd.read_parquet("../data/arf_chunks_parsed.parquet")
if df["chunk_id"].dtype != "int64":
    df["chunk_id"] = df["chunk_id"].astype(int)

def parse_relations(val):
    if val is None:
        return []
    if isinstance(val, str):
        return ast.literal_eval(val)
    return list(val)

# Real adaptive bin counts (README's own formula), not a stand-in
# number — the standalone tests used num_bins=12 as a placeholder;
# this grounds the check in what actual books actually get.
rel_counts = df.groupby("book_id")["relations"].apply(
    lambda col: sum(len(parse_relations(r)) for r in col)
)
bin_counts = rel_counts.apply(lambda n: max(6, min(20, n // 15)))
print(f"Bin count distribution across corpus:\n{bin_counts.value_counts().sort_index()}")

sparsest_book, densest_book = bin_counts.idxmin(), bin_counts.idxmax()
mid_book = bin_counts.sort_values().index[len(bin_counts) // 2]

test_phrases = [
    "near the end of the book", "in the first third", "the last quarter",
    "by the middle of the story", "early in the book", "late in the story",
    "right at the beginning", "two-thirds of the way through",
    "about halfway through", "the second half", "towards the end",
    "a third of the way through", "two thirds through the book",
    "what if the hero died in the second battle",  # must stay NO MATCH — out of scope
    "gibberish that matches nothing",              # must stay NO MATCH
]

for label, book_id in [("sparsest", sparsest_book), ("mid", mid_book), ("densest", densest_book)]:
    n_bins = bin_counts[book_id]
    print(f"\n=== {label}: book {book_id} (relations={rel_counts[book_id]}, N={n_bins}) ===")
    for p in test_phrases:
        r = parse_temporal_phrase(p)
        result = "NO MATCH" if r is None else f"bins={resolve_to_bins(r, n_bins)}"
        print(f"  {p!r:48} -> {result}")

Bin count distribution across corpus:
relations
6      7
9      1
10     1
11     1
12     1
13     1
20    84
Name: count, dtype: int64

=== sparsest: book 15284 (relations=27, N=6) ===
  'near the end of the book'                       -> bins=(np.int64(6), np.int64(6))
  'in the first third'                             -> bins=(1, 3)
  'the last quarter'                               -> bins=(5, np.int64(6))
  'by the middle of the story'                     -> bins=(1, 4)
  'early in the book'                              -> bins=(1, 1)
  'late in the story'                              -> bins=(np.int64(6), np.int64(6))
  'right at the beginning'                         -> bins=(1, 1)
  'two-thirds of the way through'                  -> bins=(4, 5)
  'about halfway through'                          -> bins=(3, 4)
  'the second half'                                -> bins=(4, np.int64(6))
  'towards the end'                                -> bins=(np.int64(6), np.int64(6))
  'a th

In [8]:
# start <= end and both within [1, N], checked across every N value
# that actually occurs in the corpus — an off-by-one at, say, N=6 or
# N=20 specifically could easily hide between the three books above.
failures = []
for book_id in bin_counts.index:
    n_bins = bin_counts[book_id]
    for p in test_phrases:
        r = parse_temporal_phrase(p)
        if r is None:
            continue
        start, end = resolve_to_bins(r, n_bins)
        if not (1 <= start <= end <= n_bins):
            failures.append((p, book_id, n_bins, start, end))

if failures:
    print(f"{len(failures)} sanity-check failures:")
    for f in failures:
        print(f"  {f}")
else:
    print(f"All checks passed across every N in the corpus ({bin_counts.min()}\u2013{bin_counts.max()}).")

All checks passed across every N in the corpus (6–20).
